# NCBI Disease Dataset Exploration

In [6]:
from datasets import load_dataset

# ncbi_disease still has a loading script on the Hub which datasets>=3 no longer
# executes. Load from the Hub's auto-generated parquet export instead.
PARQUET_BASE = "hf://datasets/ncbi_disease@refs/convert/parquet/ncbi_disease"

dataset = load_dataset(
    "parquet",
    data_files={
        "train":      f"{PARQUET_BASE}/train/0000.parquet",
        "validation": f"{PARQUET_BASE}/validation/0000.parquet",
        "test":       f"{PARQUET_BASE}/test/0000.parquet",
    },
)
print(dataset)

DatasetDict({
    train: Dataset({
        features: ['id', 'tokens', 'ner_tags'],
        num_rows: 5433
    })
    validation: Dataset({
        features: ['id', 'tokens', 'ner_tags'],
        num_rows: 924
    })
    test: Dataset({
        features: ['id', 'tokens', 'ner_tags'],
        num_rows: 941
    })
})


In [7]:
# Inspect a sample from the training split
sample = dataset["train"][0]
print("Tokens:", sample["tokens"])
print("NER tags:", sample["ner_tags"])

Tokens: ['Identification', 'of', 'APC2', ',', 'a', 'homologue', 'of', 'the', 'adenomatous', 'polyposis', 'coli', 'tumour', 'suppressor', '.']
NER tags: [0, 0, 0, 0, 0, 0, 0, 0, 1, 2, 2, 2, 0, 0]


In [8]:
# Label distribution
import pandas as pd
from collections import Counter

feat = dataset["train"].features["ner_tags"]
if hasattr(feat.feature, "names"):
    label_names = feat.feature.names
    all_tags = [label_names[t] for ex in dataset["train"] for t in ex["ner_tags"]]
else:
    all_tags = [t for ex in dataset["train"] for t in ex["ner_tags"]]

df = pd.DataFrame(Counter(all_tags).items(), columns=["label", "count"]).sort_values("label")
print(df.to_string(index=False))

    label  count
B-Disease   5145
I-Disease   6122
        O 124819


In [9]:
# Pretty print first 5 samples showing entity spans clearly
for i in range(5):
    sample = dataset["train"][i]
    tokens = sample["tokens"]
    tags = sample["ner_tags"]
    
    entities = []
    current_entity = []
    
    for token, tag in zip(tokens, tags):
        if tag == 1:  # B-Disease
            if current_entity:
                entities.append(" ".join(current_entity))
            current_entity = [token]
        elif tag == 2:  # I-Disease
            current_entity.append(token)
        else:
            if current_entity:
                entities.append(" ".join(current_entity))
                current_entity = []
    
    if current_entity:
        entities.append(" ".join(current_entity))
    
    print(f"Sample {i+1}:")
    print(f"  Text: {' '.join(tokens)}")
    print(f"  Diseases found: {entities if entities else 'None'}")
    print()

Sample 1:
  Text: Identification of APC2 , a homologue of the adenomatous polyposis coli tumour suppressor .
  Diseases found: ['adenomatous polyposis coli tumour']

Sample 2:
  Text: The adenomatous polyposis coli ( APC ) tumour - suppressor protein controls the Wnt signalling pathway by forming a complex with glycogen synthase kinase 3beta ( GSK - 3beta ) , axin / conductin and betacatenin .
  Diseases found: ['adenomatous polyposis coli ( APC ) tumour']

Sample 3:
  Text: Complex formation induces the rapid degradation of betacatenin .
  Diseases found: None

Sample 4:
  Text: In colon carcinoma cells , loss of APC leads to the accumulation of betacatenin in the nucleus , where it binds to and activates the Tcf - 4 transcription factor ( reviewed in [ 1 ] [ 2 ] ) .
  Diseases found: ['colon carcinoma']

Sample 5:
  Text: Here , we report the identification and genomic structure of APC homologues .
  Diseases found: None

